# Would a chemical-similarity split differ from the functional-group one?

`mechanism_holdout` holds out a SMARTS functional group. The worry with that split is that a
group is a *label*, not a region of chemical space: its members can be scattered, their close
analogues can stay in training, and a model may then transfer by interpolation rather than by
mechanism. The alternative is to hold out a **cluster** — a connected region of the odorant
panel — so that what is removed is a neighbourhood, not a name.

Before implementing that split, the question is whether it would even be a different
experiment. This notebook answers that, and nothing else:

1. two distance spaces over the same molecules — **Tanimoto** on ECFP4, and **cosine** on the
   GIN embeddings the pipeline actually feeds the graph;
2. clustering in each, once **non-parametrically** (HDBSCAN, which also decides how many) and
   once with **k chosen by silhouette**, so the answer does not rest on one algorithm;
3. agreement with the existing SMARTS partition — ARI/AMI plus the contingency table;
4. and the part that decides whether the change is worth making: the **isolation controls**
   (`struct_leak`, `func_redund`) computed for the candidate clusters and for the SMARTS
   classes, side by side.

Everything is computed here — this notebook is small enough that the usual
script-computes / notebook-draws split would only get in the way.

Both insect panels are done, and they are **not** the same molecules: CC and HC share only 50
of their 110 odorants, so each needs its own answer.

In [ ]:
%matplotlib inline
import pathlib, sys, itertools
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from scripts.modeling.analysis.mechanism_holdout import (
    DATASETS, load_dataset, class_members, class_targets, struct_leak, func_redund)
from orbind.dataset import load_npz_dict

DS = ["cc", "hc"]           # both insect panels
MIN_MEMBERS = 6             # a group smaller than this cannot be a holdout (pipeline value)
MAX_FRAC = 0.5              # nor can one that would take most of the panel with it
SEED = 0

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, DataStructs
RDLogger.DisableLog("rdApp.*")

def fingerprints(keys, smi):
    """ECFP4 for the molecules rdkit can parse, and the surviving keys in the same order."""
    out_k, out_f = [], []
    for o in keys:
        m = Chem.MolFromSmiles(smi.get(o, ""))
        if m is not None:
            out_k.append(o)
            out_f.append(AllChem.GetMorganFingerprintAsBitVect(m, 2, 2048))
    return out_k, out_f

def tanimoto_D(fps):
    n = len(fps)
    D = np.zeros((n, n))
    for i in range(n):
        D[i] = 1.0 - np.asarray(DataStructs.BulkTanimotoSimilarity(fps[i], fps))
    np.fill_diagonal(D, 0.0)
    return np.clip((D + D.T) / 2, 0, None)

def cosine_D(X):
    Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
    D = 1.0 - Xn @ Xn.T
    np.fill_diagonal(D, 0.0)
    return np.clip((D + D.T) / 2, 0, None)

DATA = {}
for ds in DS:
    p, recs, ods, R, smi = load_dataset(ds)
    gin = load_npz_dict(str(ROOT / "data" / DATASETS[ds]["mol"]))
    mols, fps = fingerprints([o for o in ods if o in gin], smi)
    X = np.stack([gin[o] for o in mols])
    DATA[ds] = dict(p=p, recs=recs, ods=ods, R=R, smi=smi, mols=mols,
                    D={"tanimoto": tanimoto_D(fps), "gin": cosine_D(X)})
    print(f"{ds}: {len(ods)} odorants, {len(mols)} with both a structure and a GIN vector "
          f"| receptors {len(recs)}")

for ds in DS:
    for name, D in DATA[ds]["D"].items():
        off = D[np.triu_indices(len(D), 1)]
        print(f"{ds}/{name:9} distances: median {np.median(off):.3f}  "
              f"5-95% {np.percentile(off, 5):.3f}-{np.percentile(off, 95):.3f}")

## 1 — The existing split, and how much of a partition it actually is

The SMARTS classes are **not** a partition: a molecule can be an ester and an aromatic at once,
and many odorants match nothing at all. That matters twice over — for the comparison below,
which needs a partition, and for the holdout itself, where an overlapping class means the
"removed" chemistry is only partly removed.

Watch the overlap matrix rather than the count. `alcohol` is `[#6][OX2H]`, which a carboxylic
acid's own -OH satisfies, so **every acid is also an alcohol** under these patterns. That is not
a detail: it means holding out `alcohol` removes the acids too, and holding out
`carboxylic_acid` leaves their closest hydroxyl analogues in training.

In [ ]:
PRIMARY = {}
for ds in DS:
    d = DATA[ds]
    cls = class_members(d["ods"], d["smi"], DATASETS[ds]["classes"], MIN_MEMBERS)
    order = [c for c in DATASETS[ds]["classes"] if c in cls]
    hits = {o: [c for c in order if o in cls[c]] for o in d["mols"]}
    prim = np.array([h[0] if h else "unlabelled" for h in hits.values()])
    PRIMARY[ds] = dict(cls=cls, order=order, hits=hits, primary=prim)
    n_multi = sum(len(h) > 1 for h in hits.values())
    n_none = sum(len(h) == 0 for h in hits.values())
    print(f"\n{ds}: {len(order)} classes {order}")
    print("   sizes: " + ", ".join(f"{c}={len(cls[c])}" for c in order))
    print(f"   {n_multi}/{len(hits)} molecules match MORE than one class, {n_none} match none")
    for o, h in [(o, h) for o, h in hits.items() if len(h) > 1][:3]:
        print(f"     e.g. {d['smi'][o]:38} -> {h}")
    ov = pd.DataFrame([[len(cls[a] & cls[b]) for b in order] for a in order],
                      index=order, columns=order)
    print(f"   how many of ROW are also COLUMN:")
    display(ov)

## 2 — Clustering, two spaces x two algorithms

**HDBSCAN** picks the number of clusters itself and is allowed to call a molecule noise, which
is the honest non-parametric answer: on a 110-molecule panel some odorants genuinely sit alone.
**Agglomerative** with average linkage needs a `k`, so `k` is swept and chosen by the silhouette
on the same precomputed distances — parametric, but with the choice made by the data.

Both run on both distance spaces, giving four candidate partitions per dataset. If they
disagree with each other, no single one of them is evidence about anything.

In [ ]:
from sklearn.cluster import HDBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score

def best_agglomerative(D, kmin=2, kmax=12):
    """Sweep k, keep the silhouette-best. Returns (labels, silhouette, k)."""
    best = (None, -2.0, None)
    for k in range(kmin, min(kmax, len(D) - 1) + 1):
        lab = AgglomerativeClustering(n_clusters=k, metric="precomputed",
                                      linkage="average").fit_predict(D)
        if len(set(lab)) < 2:
            continue
        s = silhouette_score(D, lab, metric="precomputed")
        if s > best[1]:
            best = (lab, s, k)
    return best

CLUSTERS, CHOICE = {}, {}
for ds in DS:
    CLUSTERS[ds] = {}
    for space, D in DATA[ds]["D"].items():
        Dd = D.astype(np.float64)
        # copy=True is NOT optional: with the default, HDBSCAN mutates the distance matrix
        # in place and everything downstream then clusters a corrupted one
        h = HDBSCAN(metric="precomputed", min_cluster_size=MIN_MEMBERS,
                    copy=True).fit_predict(Dd)
        lab, sil, k = best_agglomerative(Dd)
        CLUSTERS[ds][f"hdbscan/{space}"] = h
        CLUSTERS[ds][f"agglom/{space}"] = lab
        CHOICE[(ds, space)] = (k, sil)
        nz = h >= 0
        sizes = sorted(np.bincount(h[nz]).tolist(), reverse=True) if nz.any() else []
        print(f"{ds}/{space:9} HDBSCAN: {len(set(h[nz]))} clusters, {(~nz).sum()}/{len(h)} "
              f"noise, sizes {sizes}")
        print(f"{ds}/{space:9} agglom : k={k} by silhouette ({sil:+.3f}), sizes "
              f"{sorted(np.bincount(lab).tolist(), reverse=True)}")

## 3 — Do the clusters reproduce the functional groups?

ARI and AMI are computed against the primary-label partition. **HDBSCAN noise is excluded**
from its scores rather than counted as one big cluster, which would flatter the agreement.

The number to read is not "is it high" but "is it near 1". A high ARI means the new split is
the old one under another name, and implementing it buys nothing.

In [ ]:
from sklearn.metrics import (adjusted_rand_score, adjusted_mutual_info_score,
                             homogeneity_score, completeness_score)

rows = []
for ds in DS:
    prim = PRIMARY[ds]["primary"]
    for name, lab in CLUSTERS[ds].items():
        ok = lab >= 0
        k = len(set(lab[ok]))
        r = dict(dataset=ds, clustering=name, n_used=int(ok.sum()), k=k,
                 biggest=int(np.bincount(lab[ok]).max()) if k else 0)
        # with everything called noise there is nothing to agree ABOUT, and sklearn happily
        # returns ARI = 1.0 on two empty labellings -- which would read as perfect agreement
        agree_ok = k >= 2 and ok.sum() >= 2
        for key, fn in (("ARI", adjusted_rand_score), ("AMI", adjusted_mutual_info_score),
                        ("homogeneity", homogeneity_score),
                        ("completeness", completeness_score)):
            r[key] = fn(prim[ok], lab[ok]) if agree_ok else np.nan
        rows.append(r)
agree = pd.DataFrame(rows)
display(agree.style.format({c: "{:+.3f}" for c in
                            ["ARI", "AMI", "homogeneity", "completeness"]},
                           na_rep="-").hide(axis="index"))
print("`biggest` is the largest cluster: one holding most of the panel is not a holdout, "
      "whatever its ARI.")

ncol = len(CLUSTERS[DS[0]])
fig, axes = plt.subplots(len(DS), ncol, figsize=(4.4 * ncol, 4.0 * len(DS)), squeeze=False)
for i, ds in enumerate(DS):
    prim = PRIMARY[ds]["primary"]
    labels = PRIMARY[ds]["order"] + ["unlabelled"]
    for j, (name, lab) in enumerate(CLUSTERS[ds].items()):
        ax = axes[i, j]
        ks = sorted(set(lab))
        M = np.array([[int(((prim == c) & (lab == k)).sum()) for k in ks] for c in labels])
        ax.imshow(M, aspect="auto", cmap="Blues")
        ax.set_xticks(range(len(ks)))
        ax.set_xticklabels(["noise" if k < 0 else k for k in ks], fontsize=8)
        ax.set_yticks(range(len(labels)))
        ax.set_yticklabels(labels if j == 0 else [], fontsize=8)
        for a, b in itertools.product(range(M.shape[0]), range(M.shape[1])):
            if M[a, b]:
                ax.text(b, a, M[a, b], ha="center", va="center", fontsize=7,
                        color="white" if M[a, b] > M.max() * .6 else "black")
        ax.set_title(f"{ds} — {name}", fontsize=9)
fig.suptitle("SMARTS class (rows) x cluster (columns)")
fig.tight_layout()
plt.show()

## 4 — The part that decides it: is a cluster better isolated?

Agreement alone does not say the new split is *better*. The reason to prefer a cluster is that
it should be harder to reach from what stays in training. The pipeline already measures exactly
that, and the same two functions are reused here so the numbers are comparable to the ones in
the mechanism-holdout tables:

* **`struct_leak`** — mean best Tanimoto from a member of the held-out group to a retained
  odorant. Lower is a cleaner holdout.
* **`func_redund`** — correlation across receptors between the group's target and general
  responsiveness. Lower means the group is functionally distinct rather than "responds to
  everything".

Only groups of at least `MIN_MEMBERS` are shown: a smaller one cannot be a holdout at all.

In [ ]:
def isolation(ds, iks):
    d = DATA[ds]
    tg = class_targets(DATASETS[ds]["kind"], d["p"], d["R"], d["recs"], d["ods"],
                       list(d["recs"]), iks)
    sl = struct_leak(d["smi"], d["ods"], iks)
    fr = func_redund(d["R"], d["recs"], d["ods"], tg["order"], iks, tg["rsa_target"])
    tr = float(np.clip((1 - (sl if np.isfinite(sl) else 0.0)) *
                       (1 - (fr if np.isfinite(fr) else 0.0)), 0, 1))
    return dict(n=len(iks), struct_leak=sl, func_redund=fr, trust=tr)

rows = []
for ds in DS:
    for c in PRIMARY[ds]["order"]:
        rows.append(dict(dataset=ds, source="SMARTS", group=c, usable=True,
                         **isolation(ds, PRIMARY[ds]["cls"][c])))
    mols = np.array(DATA[ds]["mols"])
    for name, lab in CLUSTERS[ds].items():
        for k in sorted(set(lab[lab >= 0])):
            iks = set(mols[lab == k])
            if len(iks) >= MIN_MEMBERS:
                rows.append(dict(dataset=ds, source=name, group=f"c{k}",
                                 usable=bool(len(iks) <= MAX_FRAC * len(mols)),
                                 **isolation(ds, iks)))
iso = pd.DataFrame(rows)
display(iso.style.format({"struct_leak": "{:.3f}", "func_redund": "{:.3f}",
                          "trust": "{:.3f}"}).hide(axis="index"))

use = iso[iso["usable"]]
summ = (use.assign(kind=np.where(use["source"] == "SMARTS", "SMARTS", "cluster"))
           .groupby(["dataset", "kind"])[["struct_leak", "func_redund", "trust", "n"]]
           .agg(["mean", "min", "max"]).round(3))
print(f"\nSMARTS classes vs candidate clusters -- usable only "
      f"({MIN_MEMBERS} <= n <= {MAX_FRAC:.0%} of the panel):")
display(summ)

fig, axes = plt.subplots(1, len(DS), figsize=(6.0 * len(DS), 4.4), squeeze=False)
for i, ds in enumerate(DS):
    ax = axes[0, i]
    sub = iso[iso.dataset == ds]
    for kind, mark in (("SMARTS", "o"), ("cluster", "^")):
        s = sub[sub["source"] == "SMARTS"] if kind == "SMARTS" else             sub[(sub["source"] != "SMARTS") & sub["usable"]]
        ax.scatter(s["struct_leak"], s["func_redund"], marker=mark, s=40 + 3 * s["n"],
                   alpha=.75, label=kind)
    ax.set_xlabel("struct_leak (lower = cleaner)")
    ax.set_ylabel("func_redund")
    ax.set_title(f"{ds} — isolation of candidate holdouts")
    ax.legend(frameon=False)
fig.tight_layout()
plt.show()

## 5 — The map

MDS on the same distances, coloured once by functional group and once by cluster. This is the
eyeball version of the ARI above: if the two colourings look alike, the split is the same
experiment.

In [ ]:
from sklearn.manifold import MDS

fig, axes = plt.subplots(len(DS), 4, figsize=(19, 4.4 * len(DS)), squeeze=False)
for i, ds in enumerate(DS):
    prim = PRIMARY[ds]["primary"]
    labels = PRIMARY[ds]["order"] + ["unlabelled"]
    cmap = {c: plt.cm.tab10(j % 10) for j, c in enumerate(labels)}
    cmap["unlabelled"] = (.8, .8, .8, 1.)
    for j, space in enumerate(DATA[ds]["D"]):
        Y = MDS(n_components=2, dissimilarity="precomputed", random_state=SEED,
                normalized_stress="auto").fit_transform(DATA[ds]["D"][space])
        ax = axes[i, 2 * j]
        for c in labels:
            m = prim == c
            ax.scatter(Y[m, 0], Y[m, 1], s=26, color=cmap[c], label=c, alpha=.85)
        ax.set_title(f"{ds}/{space} — SMARTS class", fontsize=10)
        if i == 0 and j == 0:
            ax.legend(fontsize=7, frameon=False, ncol=2)
        lab = CLUSTERS[ds][f"hdbscan/{space}"]
        ax = axes[i, 2 * j + 1]
        ax.scatter(Y[lab < 0, 0], Y[lab < 0, 1], s=20, color="lightgray", label="noise")
        for k in sorted(set(lab[lab >= 0])):
            m = lab == k
            ax.scatter(Y[m, 0], Y[m, 1], s=26, label=f"c{k}", alpha=.85)
        ax.set_title(f"{ds}/{space} — HDBSCAN", fontsize=10)
        ax.legend(fontsize=7, frameon=False, ncol=2)
    for ax in axes[i]:
        ax.set_xticks([])
        ax.set_yticks([])
fig.tight_layout()
plt.show()

## Reading this

**If ARI is high (say above ~0.5)** the cluster split is the functional-group split wearing a
different name, and implementing it would not answer the worry that prompted it. Look then at
whether the *isolation* differs anyway — a similar partition with lower `struct_leak` would
still be a better-controlled version of the same experiment.

**If ARI is low but the isolation is no better,** the clusters are a different split that is not
a cleaner one. That is the outcome worth knowing before spending a run on it: a new split that
leaks as much teaches nothing new.

**If ARI is low and `struct_leak` drops,** the change is worth making, and this notebook has
already named the groups to hold out.

Two cautions on the numbers above. The clusters are built from the molecule side alone, so
`func_redund` — a property of the response matrix — is free to be *worse* for them than for a
functional group; a chemically tight cluster can still be one every receptor responds to. And
the four clusterings disagreeing with each other is itself a result: it would mean the panel has
no cluster structure robust enough to hang a split on, which is a reason to keep the SMARTS
split rather than to pick whichever clustering looks best.